# Fake News Detection - Phase 3: Model Evaluation and Plotting

In this notebook, we will:
1. Load the split test set (`test_data.csv`).
2. Load the trained classifier, vectorizer, and label encoder.
3. Perform inferences and calculate accuracy, precision, recall, and F1 metrics.
4. Plot the Confusion Matrix Heatmap.
5. Plot the ROC (Receiver Operating Characteristic) Curve.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_curve,
    auc,
    ConfusionMatrixDisplay
)

# Append src folder to path
sys.path.append(os.path.abspath('../src'))
from feature_extraction import load_vectorizer

## 1. Load Assets and Test Split

In [ ]:
df_test = pd.read_csv('../dataset/test_data.csv')
df_test.dropna(subset=['cleaned_content'], inplace=True)

X_test_text = df_test['cleaned_content']
y_test = df_test['label']

# Load model assets
model = joblib.load('../models/fake_news_model.pkl')
vectorizer = load_vectorizer('../models/vectorizer.pkl')
encoder = joblib.load('../models/label_encoder.pkl')

## 2. Generate Predictions

In [ ]:
# Vectorize
X_test_features = vectorizer.transform(X_test_text)

# Predict classes and probabilities
y_pred = model.predict(X_test_features)
y_probs = model.predict_proba(X_test_features)[:, 1] # Probability of True class

# Decode numeric targets back to strings
y_test_decoded = encoder.inverse_transform(y_test)
y_pred_decoded = encoder.inverse_transform(y_pred)

## 3. Print Metrics Summary

In [ ]:
print("Classification Report:")
print(classification_report(y_test_decoded, y_pred_decoded))

cm = confusion_matrix(y_test_decoded, y_pred_decoded, labels=encoder.classes_)
print("Confusion Matrix:")
print(pd.DataFrame(cm, index=[f'Actual {c}' for c in encoder.classes_], columns=[f'Predicted {c}' for c in encoder.classes_]))

## 4. Visualizations

In [ ]:
# Plot Confusion Matrix Heatmap
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=encoder.classes_, 
            yticklabels=encoder.classes_)
plt.title('Confusion Matrix Heatmap')
plt.ylabel('Actual Category')
plt.xlabel('Predicted Category')
plt.tight_layout()
plt.show()

In [ ]:
# Plot ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_probs)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC Curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.show()